# 08_h1_endpoint_packet: independent resumable A100 bundle

translated-cut endpoint packets with exact-benchmark wall orientation. This notebook launches only `08_h1_endpoint_packet`; it never continues into another experiment. The sampling layout uses 25 independent trajectories in five immutable five-trajectory shards. Run the default report-only pass first, inspect the exact verified/pending inventory, then set `RESUME_REPORT_ONLY=False` to compute only missing shards.

## 1. Mount Drive and locate the campaign package


In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, subprocess, sys

from google.colab import auth, drive
drive.mount('/content/drive', force_remount=False)

auth.authenticate_user()  # one Drive API authorization per fresh runtime

MYDRIVE = Path('/content/drive/MyDrive')
DRIVE_CAMPAIGN_ROOT = MYDRIVE / 'final_production_new_designs_v4'
EXPECTED_OPERATIONAL_RELEASE = 'v4-drivefs-independent-20260901-r2'
import os, shutil, tempfile
import google.auth, google_auth_httplib2, httplib2
from googleapiclient.discovery import build as _build_google_service
_credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
_authorized_http = google_auth_httplib2.AuthorizedHttp(_credentials, http=httplib2.Http(timeout=60))
_drive_service = _build_google_service('drive', 'v3', http=_authorized_http, cache_discovery=False)
_drive_fields = 'nextPageToken,files(id,name,parents,size,sha256Checksum,trashed,mimeType)'
def _drive_query_escape(value):
    return str(value).replace('\\', '\\\\').replace("'", "\\'")
def _drive_named(parent_id, name):
    query = f"'{_drive_query_escape(parent_id)}' in parents and name = '{_drive_query_escape(name)}' and trashed = false"
    rows, page_token = [], None
    while True:
        response = _drive_service.files().list(q=query, spaces='drive', fields=_drive_fields, pageSize=100, pageToken=page_token).execute()
        rows.extend(response.get('files', []))
        page_token = response.get('nextPageToken')
        if not page_token:
            break
    if len(rows) != 1:
        raise RuntimeError(f'Drive API expected one {name!r} below parent {parent_id}, found {len(rows)}')
    return rows[0]
def _drive_folder(parent_id, name):
    row = _drive_named(parent_id, name)
    if row.get('mimeType') != 'application/vnd.google-apps.folder':
        raise RuntimeError(f'Drive deployment component is not a folder: {name!r}')
    return row
def _drive_download_verified(row, expected=None):
    raw = _drive_service.files().get_media(fileId=str(row['id'])).execute()
    if not isinstance(raw, (bytes, bytearray)):
        raise RuntimeError(f'Drive API returned non-binary content for {row.get("name")!r}')
    raw = bytes(raw)
    actual_sha = hashlib.sha256(raw).hexdigest()
    if len(raw) != int(row.get('size', -1)) or actual_sha != row.get('sha256Checksum'):
        raise RuntimeError(f'Drive server metadata/readback mismatch for {row.get("name")!r}')
    if expected is not None and (len(raw) != int(expected['bytes']) or actual_sha != expected['sha256']):
        raise RuntimeError(f'Deployment manifest mismatch for {row.get("name")!r}')
    return raw
_campaign_row = _drive_folder('root', DRIVE_CAMPAIGN_ROOT.name)
_manifest_row = _drive_named(str(_campaign_row['id']), 'deployment_manifest.json')
_manifest_raw = _drive_download_verified(_manifest_row)
deployment_manifest = json.loads(_manifest_raw.decode('utf-8'))
if deployment_manifest.get('schema') != 'classA_v4_deployment_manifest_v1':
    raise RuntimeError('Server deployment manifest has the wrong schema')
if deployment_manifest.get('operational_release') != EXPECTED_OPERATIONAL_RELEASE:
    raise RuntimeError(f'Wrong server deployment release: {deployment_manifest.get("operational_release")!r}')
if deployment_manifest.get('campaign_parent') != DRIVE_CAMPAIGN_ROOT.name:
    raise RuntimeError('Server deployment manifest names the wrong campaign parent')
if set(deployment_manifest.get('bundles', [])) != {'01_p1_chern_dynamics', '08_h1_endpoint_packet'}:
    raise RuntimeError('Server deployment manifest has the wrong bundle set')
deployment_files = deployment_manifest.get('files', {})
if not deployment_files:
    raise RuntimeError('The v4 deployment manifest is empty or missing its file table')
if int(deployment_manifest.get('file_count', -1)) != len(deployment_files):
    raise RuntimeError('The v4 deployment manifest file count is inconsistent')
_expected_notebooks = {'01_p1_chern_dynamics/run_production_bundle.ipynb', '08_h1_endpoint_packet/run_production_bundle.ipynb'}
notebook_manifest_files = {relative: expected for relative, expected in deployment_files.items() if str(relative).endswith('.ipynb')}
if set(notebook_manifest_files) != _expected_notebooks:
    raise RuntimeError('The v4 deployment manifest has the wrong notebook set')
runtime_deployment_files = {relative: expected for relative, expected in deployment_files.items() if relative not in notebook_manifest_files}
if not runtime_deployment_files:
    raise RuntimeError('The v4 deployment manifest has no executable payload')
def _deployment_path(root, relative):
    clean = Path(str(relative))
    if clean.is_absolute() or clean == Path('.') or '..' in clean.parts:
        raise RuntimeError(f'Unsafe deployment-manifest path: {relative!r}')
    return root / clean
deployment_key = hashlib.sha256(_manifest_raw).hexdigest()[:16]
LOCAL_CAMPAIGN_ROOT = Path('/content') / f'final_production_new_designs_v4_{deployment_key}'
def _verified_deployment(root):
    for relative, expected in runtime_deployment_files.items():
        candidate = _deployment_path(root, relative)
        if not candidate.is_file() or candidate.stat().st_size != int(expected['bytes']):
            return False
        if hashlib.sha256(candidate.read_bytes()).hexdigest() != expected['sha256']:
            return False
    return True
if not _verified_deployment(LOCAL_CAMPAIGN_ROOT):
    stage_root = Path(tempfile.mkdtemp(prefix='.classA_v4_stage.', dir='/content'))
    try:
        _folder_ids = {Path('.'): str(_campaign_row['id'])}
        for relative, expected in runtime_deployment_files.items():
            clean = Path(str(relative))
            parent_id = str(_campaign_row['id'])
            walked = Path('.')
            for component in clean.parts[:-1]:
                walked = walked / component
                if walked not in _folder_ids:
                    _folder_ids[walked] = str(_drive_folder(parent_id, component)['id'])
                parent_id = _folder_ids[walked]
            remote_row = _drive_named(parent_id, clean.name)
            if remote_row.get('mimeType') == 'application/vnd.google-apps.folder':
                raise RuntimeError(f'Deployment file is unexpectedly a folder: {relative}')
            raw = _drive_download_verified(remote_row, expected)
            destination = _deployment_path(stage_root, relative)
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(raw)
            if destination.stat().st_size != int(expected['bytes']) or hashlib.sha256(destination.read_bytes()).hexdigest() != expected['sha256']:
                raise RuntimeError(f'Deployment API download failed verification: {relative}')
        (stage_root / 'deployment_manifest.json').write_bytes(_manifest_raw)
        if LOCAL_CAMPAIGN_ROOT.exists():
            shutil.rmtree(LOCAL_CAMPAIGN_ROOT)
        os.replace(stage_root, LOCAL_CAMPAIGN_ROOT)
    except BaseException:
        shutil.rmtree(stage_root, ignore_errors=True)
        raise
if not _verified_deployment(LOCAL_CAMPAIGN_ROOT):
    raise RuntimeError('Local v4 deployment failed its final hash verification')
CAMPAIGN_ROOT = LOCAL_CAMPAIGN_ROOT
print(f'[server deployment release] {EXPECTED_OPERATIONAL_RELEASE}')
print(f'[verified executable payloads] {len(runtime_deployment_files)}; notebook records are audit-only because Colab mutates open notebook outputs')
print(f'[local verified deployment] {CAMPAIGN_ROOT}')
runner = CAMPAIGN_ROOT / 'colab_bundle_runner.py'
bundle_root = CAMPAIGN_ROOT / '08_h1_endpoint_packet'
if not runner.is_file() or not (bundle_root / 'run_bundle.py').is_file():
    raise FileNotFoundError('Upload this bundle with the final_production_new_designs_v4 parent support files')
def _load_runner_module(module_path, module_name):
    module_spec = importlib.util.spec_from_file_location(module_name, module_path)
    if module_spec is None or module_spec.loader is None:
        raise ImportError(f'Cannot load runner: {module_path}')
    module = importlib.util.module_from_spec(module_spec)
    sys.modules[module_spec.name] = module
    module_spec.loader.exec_module(module)
    return module
runner_module = _load_runner_module(runner, '_colab_bundle_runner')
bundle_config = json.loads((bundle_root / 'production_config.json').read_text())
OUTPUT_BUNDLE = bundle_config.get('output_bundle', bundle_root.name)
print(f'[campaign] {CAMPAIGN_ROOT}')
print(f'[bundle] {bundle_root}')


## 2. Select this bundle's queue


In [ ]:
RUN_PROFILE = 'production'
BUNDLE = '08_h1_endpoint_packet'
CASE_PREFIXES = ()  # e.g. ('S2_',); empty runs every currently eligible case
CASE_IDS = ()       # exact IDs; empty uses prefixes/full bundle
MAX_SESSION_GPU_HOURS_OVERRIDE = None
PREFLIGHT_ONLY = False
RESUME_REPORT_ONLY = True  # inspect first; change to False to launch missing work
HEARTBEAT_SECONDS = 60
RUN_QUEUE = True
if HEARTBEAT_SECONDS <= 0:
    raise ValueError('HEARTBEAT_SECONDS must be positive')
if CASE_PREFIXES and CASE_IDS:
    raise ValueError('use CASE_PREFIXES or CASE_IDS, not both')
print(json.dumps({
    'bundle': BUNDLE, 'profile': RUN_PROFILE, 'case_prefixes': CASE_PREFIXES,
    'case_ids': CASE_IDS, 'resume_report_only': RESUME_REPORT_ONLY,
    'heartbeat_seconds': HEARTBEAT_SECONDS,
}, indent=2))


## 3. Verify server-side Drive commits

This writes a tiny probe through the Drive API, verifies its server parent, byte count, and SHA-256, then removes it. DriveFS visibility alone is never accepted as durability.


In [ ]:
RUN_REMOTE_COMMIT_PROBE = True
if RUN_REMOTE_COMMIT_PROBE:
    remote_commit_module = _load_runner_module(CAMPAIGN_ROOT / 'drive_remote_commit.py', '_colab_operational_drive_remote_commit')
    committer = remote_commit_module.DriveRemoteCommitter(drive_root=MYDRIVE)
    probe_path = MYDRIVE / bundle_config['production_output_collection'] / OUTPUT_BUNDLE / '_remote_commit_probe.json'
    probe = {'schema': 'classA_remote_commit_probe_v1', 'bundle': BUNDLE}
    remote_commit_module.publish_json(committer, probe, probe_path, replace=True, required_headroom_bytes=0)
    verified_probe = committer.path_commit_record(probe_path)
    committer.verify_record_for_path(verified_probe, probe_path)
    committer.delete_verified(verified_probe)
    print('[remote commit probe] passed and cleaned up')


## 4. Required complete-shard A100 preflight

The production queue runs or reuses this qualification automatically before its first missing shard. This optional cell executes all 80 circuit cycles and all 40 translated cuts at every observation cycle for one complete five-trajectory soft-wall, alpha_1=1 shard. A safe v4 result is archived as the real production shard zero, so the calculation is not repeated. H1-v4 records raw A100 packet-norm error, normalizes the exact-unitary packets before extracting handedness, and treats warning-level raw residuals as non-gating. It writes a version-matched memory, runtime, storage, and numerical receipt. A hard numerical failure is preserved under `_failed_qualifications` before production remains locked.


In [ ]:
RUN_A100_PREFLIGHT = False
if RUN_A100_PREFLIGHT:
    storage = runner_module.server_storage_status(drive_root=MYDRIVE, output_collection=bundle_config['production_output_collection'], working_limit_gb=12.0, absolute_edge_gb=14.0, required_headroom_gb=1.25)
    print(json.dumps(storage, indent=2, sort_keys=True))
    if not storage['clear_to_run']:
        raise RuntimeError('Server-authoritative Drive storage guard blocked H1 preflight')
    preflight_module = _load_runner_module(bundle_root / 'run_bundle.py', '_colab_h1_endpoint_preflight_runner')
    preflight_argv = ['--drive-root', str(MYDRIVE), '--mode', 'production', '--a100-preflight']
    returncode = preflight_module.main(preflight_argv)
    if returncode not in (None, 0):
        raise RuntimeError(f'Preflight runner returned nonzero status {returncode}')


## 5. Verify, resume, and stop at the end of this bundle


In [ ]:
if not RUN_QUEUE:
    raise RuntimeError('RUN_QUEUE is False')
runner_argv = [
    '--drive-root', str(MYDRIVE),
    '--bundle', BUNDLE, '--profile', RUN_PROFILE,
    '--heartbeat-seconds', str(HEARTBEAT_SECONDS),
]
for prefix in CASE_PREFIXES:
    runner_argv += ['--case-prefix', str(prefix)]
for case_id in CASE_IDS:
    runner_argv += ['--case-id', str(case_id)]
if MAX_SESSION_GPU_HOURS_OVERRIDE is not None:
    runner_argv += ['--max-session-hours', str(MAX_SESSION_GPU_HOURS_OVERRIDE)]
if PREFLIGHT_ONLY:
    runner_argv.append('--preflight-only')
if RESUME_REPORT_ONLY:
    runner_argv.append('--resume-report-only')
command = [sys.executable, '-u', str(runner), *runner_argv]
production_collection = bundle_config.get('production_output_collection', 'classA_final_production_outputs/production_10sample_v4_occupied_frame_cycle_resolved')
pilot_collection = bundle_config.get('pilot_output_collection', 'classA_pilot_outputs/production_10sample_v4_occupied_frame_cycle_resolved')
collection = production_collection if RUN_PROFILE == 'production' else pilot_collection
output_root = MYDRIVE / collection
session_dir = runner_module.local_session_root(output_collection=collection, bundle=BUNDLE)
build_id = hashlib.sha256(runner.read_bytes()).hexdigest()[:16]
print('[bundle dashboard]')
print(json.dumps({
    'bundle': BUNDLE, 'profile': RUN_PROFILE, 'output_root': str(output_root),
    'runner_build_id': build_id, 'heartbeat_seconds': HEARTBEAT_SECONDS,
    'session_dir': str(session_dir), 'case_prefixes': CASE_PREFIXES,
    'case_ids': CASE_IDS, 'resume_report_only': RESUME_REPORT_ONLY,
}, indent=2))
print(f'[launch bundle] {" ".join(command)}', flush=True)
try:
    returncode = runner_module.main(runner_argv)
    if returncode not in (None, 0):
        raise RuntimeError(f'Runner returned nonzero status {returncode}')
except BaseException:
    try:
        sessions = sorted(session_dir.glob(f'*_{BUNDLE}_*.json'), key=lambda p: p.stat().st_mtime, reverse=True)
        if sessions:
            latest = json.loads(sessions[0].read_text())
            print('[saved bundle failure: local telemetry]')
            print(json.dumps({'session': str(sessions[0]), 'failure': latest.get('failure'), 'current': latest.get('current'), 'log': latest.get('session_log'), 'telemetry_errors': latest.get('telemetry_errors')}, indent=2))
    except Exception as diagnostic_error:
        print(f'[local failure diagnostics unavailable] {type(diagnostic_error).__name__}: {diagnostic_error}')
    raise


## 6. Merge 11 server-verified H1-v3 shards with 9 H1-v4 shards


In [ ]:
RUN_H1_ENDPOINT_ANALYSIS = False
if RUN_H1_ENDPOINT_ANALYSIS:
    subprocess.run([sys.executable, '-u', str(CAMPAIGN_ROOT / 'server_verified_analysis.py'), '--campaign-root', str(CAMPAIGN_ROOT), '--drive-root', str(MYDRIVE), '--bundle', BUNDLE], check=True)


## Final. Disconnect this completed Colab runtime


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
